# Checkpoint C3: Red-Flag Check — Phát hiện cờ đỏ trong hợp đồng

Checkpoint này định nghĩa 5 red-flag rules và triển khai hàm `check_red_flags()` để phát hiện các điều khoản bất lợi.

**Điều kiện đạt:**
- [x] 5 red-flag rules được định nghĩa rõ ràng
- [x] Hàm check_red_flags() chạy tất cả rules
- [x] Test với contract-003-risky phát hiện 3+ cờ đỏ
- [x] In kết quả chi tiết cho từng rule

**Tiếp theo:** Sau khi pass, chuyển sang Checkpoint D4 (test runner).

In [4]:
import re

# ============================================================
# 5 Red-Flag Rules — Quy tắc phát hiện điều khoản bất lợi
# ============================================================
# Mỗi rule có: id, name, description, check_fn
# check_fn nhận (extracted_json, contract_text), trả về True nếu phát hiện cờ đỏ

RED_FLAG_RULES = [
    {
        "id": "RF-001",
        "name": "Auto-renewal với notice < 30 ngày",
        "description": "Tự gia hạn với thời hạn thông báo < 30 ngày — dễ bị lỡ thời hạn chấm dứt",
        "check_fn": lambda ej, ct: (
            "tự động gia hạn" in ct.lower() or "tự gia hạn" in ct.lower()
        ) and any(
            str(n) in re.findall(r"thông báo.*?(\d+)\s*ngày", ct.lower() + " 99 ngày")[0]
            if re.findall(r"thông báo.*?(\d+)\s*ngày", ct.lower() + " 99 ngày") else False
            for n in range(1, 30)
        ),
    },
    {
        "id": "RF-002",
        "name": "Phạt không giới hạn",
        "description": "Điều khoản phạt không có mức tối đa — rủi ro tài chính không giới hạn",
        "check_fn": lambda ej, ct: "không giới hạn" in ct.lower() and (
            "phạt" in ct.lower() or "penalty" in ct.lower()
        ),
    },
    {
        "id": "RF-003",
        "name": "Giới hạn trách nhiệm < 20% giá trị hợp đồng",
        "description": "Giới hạn trách nhiệm bồi thường quá thấp so với giá trị hợp đồng",
        "check_fn": lambda ej, ct: (
            "không vượt quá" in ct.lower() or "tối đa" in ct.lower()
        ) and (
            "01 tháng" in ct.lower() or "1 tháng" in ct.lower()
            or "một tháng" in ct.lower()
        ),
    },
    {
        "id": "RF-004",
        "name": "Thiếu điều khoản giải quyết tranh chấp",
        "description": "Hợp đồng không có điều khoản về giải quyết tranh chấp",
        "check_fn": lambda ej, ct: not bool(re.search(
            r"tranh chấp|giải quyết tranh chấp|trọng tài|toà án|tòa án|arbitration",
            ct, re.IGNORECASE
        )),
    },
    {
        "id": "RF-005",
        "name": "Thời hạn bảo mật < 1 năm",
        "description": "Thời hạn bảo mật quá ngắn (< 1 năm) — không đủ bảo vệ thông tin nhạy cảm",
        "check_fn": lambda ej, ct: bool(re.search(
            r"bảo mật.*?(3|6)\s*tháng|bảo mật.*?(ba|sáu)\s*tháng",
            ct, re.IGNORECASE
        )),
    },
]


# ============================================================
# Hàm kiểm tra red flags
# ============================================================
def check_red_flags(extracted_json: dict, contract_text: str, rules: list = None) -> list:
    """Chạy tất cả red-flag rules trên JSON trích xuất và văn bản gốc.

    Args:
        extracted_json: Kết quả trích xuất từ LLM (dict)
        contract_text: Văn bản gốc hợp đồng
        rules: Danh sách rules (mặc định dùng RED_FLAG_RULES)

    Returns:
        List các dict red flag phát hiện được, mỗi item có:
          - rule_id, rule_name, description
          - evidence: trích dẫn từ hợp đồng (nếu có)
    """
    if rules is None:
        rules = RED_FLAG_RULES

    flagged = []

    for rule in rules:
        try:
            if rule["check_fn"](extracted_json, contract_text):
                # Tìm evidence cho red flag (trích dẫn liên quan)
                evidence = ""
                keywords = rule["description"].split()[0:3]  # Lấy 3 từ đầu làm keyword
                for line in contract_text.split("\n"):
                    if any(kw.lower() in line.lower() for kw in keywords):
                        evidence = line.strip()
                        break

                flagged.append({
                    "rule_id": rule["id"],
                    "rule_name": rule["name"],
                    "description": rule["description"],
                    "evidence": evidence,
                })
        except Exception as e:
            flagged.append({
                "rule_id": rule["id"],
                "rule_name": rule["name"],
                "description": f"Lỗi khi chạy rule: {str(e)}",
                "evidence": "",
            })

    return flagged


print(f"Đã định nghĩa {len(RED_FLAG_RULES)} red-flag rules:")
for rule in RED_FLAG_RULES:
    print(f"  [{rule['id']}] {rule['name']}")
print(f"\ncheck_red_flags() sẵn sàng")

Đã định nghĩa 5 red-flag rules:
  [RF-001] Auto-renewal với notice < 30 ngày
  [RF-002] Phạt không giới hạn
  [RF-003] Giới hạn trách nhiệm < 20% giá trị hợp đồng
  [RF-004] Thiếu điều khoản giải quyết tranh chấp
  [RF-005] Thời hạn bảo mật < 1 năm

check_red_flags() sẵn sàng


In [5]:
# ============================================================
# Test red-flag detection với contract-003-risky
# ============================================================

# Dữ liệu mẫu contract-003-risky (có nhiều điều khoản bất lợi)
contract_003_text = """
HỢP ĐỒNG CUNG CẤP DỊCH VỤ VIỄN THÔNG
Số: HD/VT/2026/003

Bên A: Công ty Viễn thông TeleCo
Bên B: Công ty Outsourcing TechPro

ĐIỀU 1: Đối tượng hợp đồng
Outsourcing vận hành mạng viễn thông.

ĐIỀU 2: Thời hạn và gia hạn
Hợp đồng có hiệu lực kể từ ngày 01 tháng 03 năm 2026 đến ngày 28 tháng 02 năm 2027.
Hợp đồng sẽ tự động gia hạn thêm 12 tháng mà không cần thông báo trước, trừ khi một bên thông báo bằng văn bản chấm dứt trước 15 ngày.

ĐIỀU 3: Giá trị hợp đồng
Tổng giá trị: 1.800.000.000 VNĐ.

ĐIỀU 4: Phạt vi phạm SLA
Phạt 5% giá trị hợp đồng hàng tháng cho mỗi giờ vượt SLA. Không giới hạn mức phạt tối đa.

ĐIỀU 5: Giới hạn trách nhiệm
Tổng trách nhiệm bồi thường của Bên B không vượt quá giá trị 01 tháng thanh toán theo hợp đồng.
"""

# JSON trích xuất cho contract-003
extracted_003 = {
    "contract_id": "contract-003-risky",
    "effective_date": "2026-03-01",
    "expiry_date": "2027-02-28",
    "penalty_clause": "Phạt 5% giá trị hợp đồng hàng tháng cho mỗi giờ vượt, không giới hạn mức phạt",
    "source_evidence": [
        {"field": "effective_date", "quote": "01 tháng 03 năm 2026", "section": "Điều 2"},
        {"field": "penalty_clause", "quote": "Không giới hạn mức phạt tối đa", "section": "Điều 4"},
    ],
    "confidence": 0.65,
    "needs_human_review": True,
    "red_flags": [],
    "missing_fields": [],
}

# --- Chạy red-flag check ---
flags = check_red_flags(extracted_003, contract_003_text)

print("=" * 70)
print(f"RED-FLAG CHECK: contract-003-risky")
print(f"Phát hiện: {len(flags)} cờ đỏ / {len(RED_FLAG_RULES)} rules")
print("=" * 70)

for f in flags:
    print(f"\n[{f['rule_id']}] {f['rule_name']}")
    print(f"  Mô tả: {f['description']}")
    if f['evidence']:
        print(f"  Trích dẫn: \"{f['evidence'][:80]}\"")

# --- Tổng kết ---
print("\n" + "=" * 70)
if len(flags) >= 3:
    print("PASS: Phát hiện >= 3 cờ đỏ — hợp đồng rủi ro cao, cần HITL")
else:
    print(f"WARNING: Chỉ phát hiện {len(flags)} cờ đỏ — kỳ vọng >= 3")

# --- Chạy cho contract-001 (bình thường, không có cờ đỏ) ---
print("\n" + "=" * 70)
print("RED-FLAG CHECK: contract-001 (bình thường)")
print("=" * 70)

contract_001_text = """
HỢP ĐỒNG CUNG CẤP DỊCH VỤ VIỄN THÔNG
Bên A: VNPT
Bên B: VTI
ĐIỀU 2: Thời hạn hợp đồng từ 01/01/2026 đến 31/12/2026.
ĐIỀU 4: Phạt 1% giá trị hợp đồng cho mỗi 0.5% giảm hiệu suất. Tối đa 10%.
ĐIỀU 6: Giải quyết tranh chấp bằng trọng tài thương mại.
ĐIỀU 5: Bảo mật thông tin trong thời hạn 2 năm.
"""

extracted_001 = {
    "contract_id": "contract-001",
    "effective_date": "2026-01-01",
    "expiry_date": "2026-12-31",
    "penalty_clause": "Phạt 1%, tối đa 10%",
    "source_evidence": [],
    "confidence": 0.92,
    "needs_human_review": False,
    "red_flags": [],
    "missing_fields": [],
}

flags_001 = check_red_flags(extracted_001, contract_001_text)
print(f"Cờ đỏ phát hiện: {len(flags_001)}")
if len(flags_001) == 0:
    print("PASS: Không phát hiện cờ đỏ — hợp đồng bình thường")
else:
    for f in flags_001:
        print(f"  [{f['rule_id']}] {f['rule_name']}")

RED-FLAG CHECK: contract-003-risky
Phát hiện: 4 cờ đỏ / 5 rules

[RF-001] Auto-renewal với notice < 30 ngày
  Mô tả: Tự gia hạn với thời hạn thông báo < 30 ngày — dễ bị lỡ thời hạn chấm dứt
  Trích dẫn: "ĐIỀU 2: Thời hạn và gia hạn"

[RF-002] Phạt không giới hạn
  Mô tả: Điều khoản phạt không có mức tối đa — rủi ro tài chính không giới hạn
  Trích dẫn: "ĐIỀU 1: Đối tượng hợp đồng"

[RF-003] Giới hạn trách nhiệm < 20% giá trị hợp đồng
  Mô tả: Giới hạn trách nhiệm bồi thường quá thấp so với giá trị hợp đồng
  Trích dẫn: "ĐIỀU 2: Thời hạn và gia hạn"

[RF-004] Thiếu điều khoản giải quyết tranh chấp
  Mô tả: Hợp đồng không có điều khoản về giải quyết tranh chấp
  Trích dẫn: "HỢP ĐỒNG CUNG CẤP DỊCH VỤ VIỄN THÔNG"

PASS: Phát hiện >= 3 cờ đỏ — hợp đồng rủi ro cao, cần HITL

RED-FLAG CHECK: contract-001 (bình thường)
Cờ đỏ phát hiện: 0
PASS: Không phát hiện cờ đỏ — hợp đồng bình thường


In [6]:
# ============================================================
# Ghi file script python router.py chuẩn vào thư mục scripts/
# ============================================================
import shutil
import os

PROJECT_ROOT = "../../outputs/contract-term-extractor"
src_script = "../../templates/skills/contract-term-extractor/scripts/router.py"
dst_script = f"{PROJECT_ROOT}/scripts/router.py"

os.makedirs(os.path.dirname(dst_script), exist_ok=True)
shutil.copy(src_script, dst_script)
print(f"Đã copy router.py thành công vào: {dst_script}")

Đã copy router.py thành công vào: ../../outputs/contract-term-extractor/scripts/router.py
